In [ ]:
import importlib
import pandas as pd
import numpy as np
from pyproj import Transformer
from pathlib import Path
import aux_funciones as aux
importlib.reload(aux)

In [ ]:
# Path de entrada
Path_DXF             = r"Avance_04-05-2026.dxf"
Path_BLOQUES         = r"MB04-26-001.csv"
Path_DEM             = r"Bronce.tif"

# Directorio de salida
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Path_ANTENA          = OUTPUT_DIR / "Antena.gpkg"
Path_GeoRef_AUTO     = OUTPUT_DIR / "DXF_Georreferenciado_Auto.dxf"
Path_GeoRef_MANUAL   = OUTPUT_DIR / "DXF_Georreferenciado_Manual.dxf"
Path_Simulador_csv   = OUTPUT_DIR / "bloques_mina_2000_SW.csv"
Path_BLOQUES_QGIS    = OUTPUT_DIR / "M_Bloques.gpkg"

# Coordenadas Antena
lat_antena = -33.1515680
lon_antena = -70.2873058

# Coordenadas Bloque (Antena)
easting_mina = 100631.25     # sitio SW; el centro se ajusta al bloque de superficie
northing_mina = 99118.75     # (celda "Bloque de superficie como centro de la geometría")
elevation_mina = None        # la calcula aux.bloque_superficie

# Dimensiones del Bloque
dx = 12.5   # Dimensión del bloque en el eje X (Metros)
dy = 12.5   # Dimensión del bloque en el eje Y (Metros)
dz = 15.0   # Dimensión del bloque en el eje Z (Metros)

# Tamaño del entorno: n_bloques hacia cada lado del bloque central -> 2n·dx x 2n·dy x 2n·dz (entre centros)
n_bloques = 80   # ~2 km en planta (entorno de ejemplo)

# Datos para simulador
label_r   = "magnetico"
sigma_r   = 0.0001   # roca constante (sólo si sigma_log = False)
eps_r     = 10
mu_r      = 1

# Conductividad de la roca logarítmica en z: sigma_techo en el bloque de roca más alto,
# sigma_fondo en el más bajo y log10(sigma) lineal en z entre ambos
sigma_log   = True
sigma_techo = 1e-5
sigma_fondo = 1e-1

label_air = "aire"
sigma_air = 0
eps_air   = 1
mu_air    = 1

## Ajuste automático DXF vs DEM

Se trata de encontrar una transformacion 2D aproximada que minimice la diferencia entre el `Z` del DXF y la altura del DEM.

In [ ]:
resultado_libre = aux.ejecutar_ajuste_libre(dxf_path=Path_DXF, 
                                            dem_path=Path_DEM
                                           )

In [ ]:
aux.exportar_dxf_georreferenciado(coef=resultado_libre['coef'], 
                                  input_path=Path_DXF, 
                                  output_path=Path_GeoRef_AUTO
                                 )

## Ajuste manual-automático DXF vs DEM

Se identifica un punto de referencia (centro) para ajustar de manera manual las poli-lineas y el DEM

In [ ]:
#Coordenadas locales Mina
aux.plot_centro_dxf(dxf_path=Path_DXF)

In [ ]:
## Se extrae centro desde QGIS según referencia
resultado_centro_fijado = aux.ejecutar_ajuste_centro_fijado(lat=-33.1523702,
                                                            lon=-70.2897360, 
                                                            dxf_path=Path_DXF, 
                                                            dem_path=Path_DEM
                                                           )

In [ ]:
aux.exportar_dxf_georreferenciado(coef=resultado_centro_fijado['coef'], 
                                  input_path=Path_DXF, 
                                  output_path=Path_GeoRef_MANUAL
                                 )

## Bloque de superficie como centro de la geometría

In [ ]:
# Centro de la geometría: bloque de roca más alto (bloque de superficie) en la columna más
# cercana a (easting_mina, northing_mina). Se verifica que los 8 bloques vecinos no sean más
# altos, para que la bobina horizontal no atraviese roca.
sitio = aux.bloque_superficie(
    csv_path=Path_BLOQUES,
    dem_path=Path_DEM,
    coef=resultado_centro_fijado['coef'],
    x=easting_mina,
    y=northing_mina,
    coord_type="local",
    dx=dx,
    dy=dy,
    dz=dz,
    n_bloques=n_bloques
)

if not sitio["plano"]:
    raise ValueError("La vecindad 3x3 del bloque central no está libre: elegir otro punto.")

easting_mina   = sitio["easting"]
northing_mina  = sitio["northing"]
elevation_mina = sitio["elevation"]

# La capa de la antena para QGIS (Antena.gpkg) marca la bobina en el centro elegido
lat_antena, lon_antena = sitio["lat"], sitio["lon"]

## Exportar capa con información del CSV para QGIS

In [ ]:
gdf_bloques = aux.exportar_bloques_identificar(coef=resultado_centro_fijado['coef'], 
                                               csv_path=Path_BLOQUES, 
                                               output_gpkg=Path_BLOQUES_QGIS, 
                                               layer_name="Modelo_Bloques"
                                              )

## Generar capa con información de la Antena

In [ ]:
# 1. Transformar de Lat/Lon a UTM Zona 19 Sur (EPSG:32719)  always_xy=True asegura que el orden sea (Longitud, Latitud) -> (X, Y)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32719", always_xy=True)
utm_easting, utm_northing = transformer.transform(lon_antena, lat_antena)
print(f"Coordenadas UTM calculadas: Easting={utm_easting:.2f}, Northing={utm_northing:.2f}")

# 2. Exportar la antena usando el tipo de coordenada "global"
gdf_antena = aux.exportar_antena_con_halos(
    x=utm_easting,                           # X en metros (UTM)
    y=utm_northing,                          # Y en metros (UTM)
    output_gpkg=Path_ANTENA,
    layer_name="antena_coordenada_global",
    halos_radios_m=[50, 100, 150, 200],      # Halos a 50m, 100m, 150m y 200m
    coord_type="global",                     # Indicamos que ya está en el sistema final
    coef=None                                # No necesitamos coeficientes de la mina aquí
)

## Extracción de características de 1 bloque

In [ ]:
# 1. Ejecutar la búsqueda directa
bloque_encontrado = aux.consultar_bloque_especifico(
    csv_path=Path_BLOQUES, 
    x=easting_mina,
    y=northing_mina,
    z=elevation_mina,
    search_radius_m=100,  
    coord_type="local"     
)

# 2. Imprimir las características
if not bloque_encontrado.empty:
    print("\n✅ ¡Bloque encontrado!")
    print("-----------------------------------")
    print(f"Distancia real al centroide: {bloque_encontrado['DISTANCIA_M'].iloc[0]:.2f} metros")
    print(f"Cobre (CUTMP): {bloque_encontrado['CUTMP'].iloc[0]} %")
    print(f"Oro (AU): {bloque_encontrado['AU'].iloc[0]} g/t")
    print(f"Zinc (ZN): {bloque_encontrado['ZN'].iloc[0]}")
else:
    print("\n❌ No se encontró ningún bloque. Intenta aumentar un poco el 'search_radius_m'.")

## Extracción grupo de Bloques (n_bloques=x)

In [ ]:
bloques_vecinos = aux.extraer_vecindad_bloques(
    csv_path=Path_BLOQUES,
    x_centro=easting_mina, 
    y_centro=northing_mina, 
    z_centro=elevation_mina,   
    n_bloques=n_bloques,
    dx=dx, 
    dy=dy, 
    dz=dz 
)

## Verificar si bloques fueron explotados (roca/aire)

In [ ]:
bloques_auditados = aux.clasificar_bloques_topografia(
    df_bloques=bloques_vecinos,
    path_dem=Path_DEM,
    coef=resultado_centro_fijado['coef'],
    dz=dz
)

## Generar .csv para Simulador en base a Modelo de Bloques

In [ ]:
# 1. Inicializar el nuevo DataFrame calculando el desfase (coordenadas relativas)
Simulador = pd.DataFrame()
Simulador['x'] = bloques_auditados['EASTING'] - easting_mina
Simulador['y'] = bloques_auditados['NORTHING'] - northing_mina
Simulador['z'] = bloques_auditados['ELEVATION'] - elevation_mina

# 2. Calcular variables eléctricas
Simulador['material_label'] = np.where(bloques_auditados['ESTA_EN_AIRE'], label_air , label_r)
Simulador['sigma']          = np.where(bloques_auditados['ESTA_EN_AIRE'], sigma_air , sigma_r)
Simulador['eps_r']          = np.where(bloques_auditados['ESTA_EN_AIRE'],   eps_air ,   eps_r)
Simulador['mu_r']           = np.where(bloques_auditados['ESTA_EN_AIRE'],    mu_air ,    mu_r)

# 2b. Conductividad de la roca logarítmica en z: sigma_fondo en el bloque de roca más bajo,
#     sigma_techo en el más alto
if sigma_log:
    es_roca = Simulador['material_label'] == label_r
    z_roca  = Simulador.loc[es_roca, 'z']
    t = (z_roca - z_roca.min()) / (z_roca.max() - z_roca.min())     # 0 en el fondo, 1 en el techo
    Simulador.loc[es_roca, 'sigma'] = sigma_fondo * (sigma_techo / sigma_fondo) ** t
    print(f"Conductividad roca: {sigma_fondo:g} S/m (z = {z_roca.min():g} m) -> "
          f"{sigma_techo:g} S/m (z = {z_roca.max():g} m), logarítmica en z")

# 3. Inspección del nuevo conjunto de datos
print("--- NUEVO DATAFRAME DE COORDENADAS RELATIVAS ---")
print(Simulador)

# 4. Guardar .csv para Simulador
Simulador.to_csv(Path_Simulador_csv, index=False)

## Visualizar Bloques: Mineral/Aire

In [ ]:
# 1. Separar los bloques reales (subterráneos) de los vacíos (en el aire)
bloques_subterraneos = bloques_auditados[~bloques_auditados["ESTA_EN_AIRE"]]
bloques_en_el_aire = bloques_auditados[bloques_auditados["ESTA_EN_AIRE"]]

# 2. Llamar a la función
aux.visualizar_modelo_bloques(
    df_mineral=bloques_subterraneos, 
    df_vacios=bloques_en_el_aire,
    col_ley='CUTMP',
    dx=dx, 
    dy=dy, 
    dz=dz
)

## Exportar Bloques para visualización en QGIS

In [ ]:
# Exportar solo los bloques reales (roca)
gdf_resultado = aux.exportar_bloques_a_gpkg(
    df_bloques=bloques_subterraneos,
    coef=resultado_centro_fijado['coef'],
    ruta_salida=Path_BLOQUES_QGIS,
    layer_name="bloques_cutmp"
)